In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [ ]:
!conda config --set channel_priority strict
!conda update -n base -c conda-forge conda
!pip install biopython
!conda install -c conda-forge ncbi-datasets-cli
!pip install cmapPy --upgrade

Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 25.3.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 25.3.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd drive
%cd MyDrive
%ls

Mounted at /content/drive
/content/drive
/content/drive/MyDrive
'=1.2.0'                  GeneClassesCloud.py        HSV1_input_fasta.fasta
'=2.16.0'                 GeneClasses.py             HSVReferenceGenome/
'=2.2.0'                  GeneLit/                   intermediate/
 301Proj/                 gene_system_pHMMer.ipynb   miRNA_prop_preprecessing/
'=3.7,'                   Genewriter.AI.gdoc         out/
 alignedHSV1              HerpetiQRMafft.ipynb       prepData.ipynb
 BVBRC_genome.csv         HerpetiQRmiRNAPrep.ipynb   __pycache__/
'Colab Notebooks'/        HerpetiQRpHMM.ipynb        RefGenes/
 condacolab_install.log   HSV1AllelesHMM/            secondPassPlusEnergetics/
 DataForWill/             HSV1_Genomes/              Standards/
 exonTraining/            HSV1_Genomes_Aligned/      WorkingFolders/
 firstPassPlusSpliceAI/   HSV1_Genomes_Strain/


In [ ]:
import os
import sys
import cmapPy
from time import sleep
import re
import pandas as pd
import pickle
from collections import defaultdict
from io import BytesIO, StringIO
from pathlib import Path
from zipfile import ZipFile
#from GeneClassesCloud import NaturalGene, Isoform, ProteinObj, SyntheticGene, IsoformGeneBody, RareCodonAnalysis, CodonAnalysis, GCAnalysis
%run GeneClassesCloud.ipynb
import cmapPy
from cmapPy.set_io.grp import read
import numpy as np
from Bio import Entrez
from Bio import SeqIO
from Bio import pairwise2
from Bio.SeqRecord import SeqRecord
import json
import shutil
import dataclasses
from EIModelsDataSourcing import downloadGenePackagesAndProcess, list_genes, prune_genes


ModuleNotFoundError: No module named 'EIModelsDataSourcing'

In [ ]:
#Get the organism name from argv
organism = sys.argv[0]
heuristics = sys.argv[1]
refgenespth = os.path.join('generider_refgenes', organism)
if not os.path.exists('generider_refgenes'):
  os.mkdir('generider_refgenes')
if not os.path.exists(refgenespth):
  os.mkdir(refgenespth)
hlist = []
for h in os.listdir(refgenespth):
  hlist.append(h)
print("Heuristics requested: ", heuristics)
print("Heuristics saved: ", hlist)
calclist = [x for x in heuristics if x not in hlist]

#Utils

In [ ]:
def loadTranscriptomeGeneBody(org: str):
    files = []
    root = os.path.join(r'generider_refgenes', org)
    print(root)

    for r, d, f in os.walk(root):
        for file in f:
            if '.pkl' in file:
                files.append(os.path.join(r, file))
    genes = []
    for f in files:
        with open(f, 'rb') as infile:
            gene = pickle.load(infile)
            genes.append(gene)
    return genes

def codonVecFromGenSeqGB(iso: IsoformGeneBody):
    '''Give it an IsoformGeneBody object and get a list of codons and tags describing where they were found'''
    #Take an isoform object and return a list of codons with tags
    codingSeq = iso.codingSeq
    taggedGeneBody = []
    inds = []
    taggedCodons = []
    for seq, loc in iso.geneBody:
        if "EXON" or "+50BPUPSTREAM" or "-50BPDOWNSTREAM" in loc.upper():
            inds.append(len(seq))
            taggedGeneBody.append([seq, loc])

    #Align the coding sequence with the gene body
    startInd = iso.fullSequence.find(codingSeq)

    #Locate start within gene body
    running = 0
    exNum = 0
    adjStart = 0
    for i in range(len(inds)):
        running += inds[i]
        exNum = i
        if running >= startInd:
            adjStart = startInd - (running - inds[i])
            break

    #Locate start of coding seq in tagged gene bodies
    flip = False
    done = False
    rem = 0
    for i in range(len(taggedGeneBody)):
        seq, loc = taggedGeneBody[i]
        if "EXON" and str(exNum + 1) in loc.upper():
            while adjStart < len(seq):
                try:
                    c = seq[adjStart: adjStart+3]
                    taggedCodons.append([c, loc])
                except IndexError:
                    try:
                        codFrag = seq[adjStart:adjStart+2]
                        rem = 1
                        c = codFrag + taggedGeneBody[i+1][0][:rem]
                        taggedCodons.append([c, "Splice"])
                        flip = True
                        break
                    except IndexError:
                        try:
                            codFrag = seq[adjStart:adjStart+1]
                            rem = 2
                            c = codFrag + taggedGeneBody[i+1][0][:rem]
                            taggedCodons.append([c, "Splice"])
                            flip = True
                            break
                        except IndexError:
                            flip = True
                            break
                if getAA(c) == '*':
                    done = True
                    break
                adjStart += 3
        if done:
            break
        adjStart = rem
        if flip:
            while adjStart < len(seq):
                try:
                    c = seq[adjStart: adjStart+3]
                    taggedCodons.append([c, loc])
                except IndexError:
                    try:
                        c = seq[adjStart:adjStart+2]
                        rem = 1
                        taggedCodons.append([c + taggedGeneBody[i+1][0][:rem], "Splice"])
                        break
                    except IndexError:
                        try:
                            c = seq[adjStart:adjStart+1]
                            rem = 2
                            taggedCodons.append([c + taggedGeneBody[i + 1][0][:rem], "Splice"])
                            break
                        except IndexError:
                            break
                if getAA(c) == '*':
                    done = True
                    break
                adjStart += 3
            if done:
                break

    predAA = ""
    for elem in taggedCodons:
        predAA = predAA + getAA(elem[0])
    aaSeq = iso.associatedProtein.aaSeq
    assert predAA == aaSeq

    print("Tagged Gene Body: ", taggedGeneBody)
    print("Tagged Codons: ", taggedCodons)

    return taggedCodons

def getAA(codon: str):
    codon = codon.upper()
    codontab = {'TCA': 'S', 'TCC': 'S', 'TCG': 'S', 'TCT': 'S', 'TTC': 'F', 'TTT': 'F', 'TTA': 'L', 'TTG': 'L',
                'TAC': 'Y', 'TAT': 'Y', 'TAA': '*', 'TAG': '*', 'TGC': 'C', 'TGT': 'C', 'TGA': '*', 'TGG': 'W',
                'CTA': 'L', 'CTC': 'L', 'CTG': 'L', 'CTT': 'L', 'CCA': 'P', 'CCC': 'P', 'CCG': 'P', 'CCT': 'P',
                'CAC': 'H', 'CAT': 'H', 'CAA': 'Q', 'CAG': 'Q', 'CGA': 'R', 'CGC': 'R', 'CGG': 'R', 'CGT': 'R',
                'ATA': 'I', 'ATC': 'I', 'ATT': 'I', 'ATG': 'M', 'ACA': 'T', 'ACC': 'T', 'ACG': 'T', 'ACT': 'T',
                'AAC': 'N', 'AAT': 'N', 'AAA': 'K', 'AAG': 'K', 'AGC': 'S', 'AGT': 'S', 'AGA': 'R', 'AGG': 'R',
                'GTA': 'V', 'GTC': 'V', 'GTG': 'V', 'GTT': 'V', 'GCA': 'A', 'GCC': 'A', 'GCG': 'A', 'GCT': 'A',
                'GAC': 'D', 'GAT': 'D', 'GAA': 'E', 'GAG': 'E', 'GGA': 'G', 'GGC': 'G', 'GGG': 'G', 'GGT': 'G'}
    return codontab[codon]

#Genes in Organism

In [ ]:
#Pull references if you don't have an example genome already
if not os.path.exists('generider_refgenes'):
  os.mkdir('generider_refgenes')
pthtogenes = 'generider_refgenes'
fullgenes = list_genes()
prunedgenes = prune_genes(pthtogenes)
downloadGenePackagesAndProcess(prunedgenes, r'generider_refgenes')


#RareCodons

In [ ]:
def defineRareCodonsRefGenes(org):
    '''Define rare codons and track where they occur in a transcript.
    Reveives: An organism name to search for downloaded refgenes
    Returns:  rareCodons, the list of codons with under 10% usage for their amino acid.
              AAFreqs, the chance of seeing any amino acid. [aa, aaFreq]
              codonCounter, a dictionary with keys=codons and vals = # of incidences/totalcodons
              totRefCods, the total number of codon
    '''

    refgenespth = os.path.join('generider_refgenes', org)
    refGenes = loadTranscriptomeGeneBody(refgenespth)
    blank = {'+50bpUpstream': 0, '-50bpDownstream': 0, 'Exon': 0, '-50bpExon': 0, '+50bpExon': 0, 'Splice': 0}
    #List of codons
    codonTable = {'AAA', 'AAT', 'AAG', 'AAC', 'ATA', 'ATT', 'ATG', 'ATC', 'AGA', 'AGT', 'AGG', 'AGC', 'ACA', 'ACT',
                  'ACG', 'ACC', 'TAA', 'TAT', 'TAG', 'TAC', 'TTA', 'TTT', 'TTG', 'TTC', 'TGA', 'TGT', 'TGG', 'TGC',
                  'TCA', 'TCT', 'TCG', 'TCC', 'GAA', 'GAT', 'GAG', 'GAC', 'GTA', 'GTT', 'GTG', 'GTC', 'GGA', 'GGT',
                  'GGG', 'GGC', 'GCA', 'GCT', 'GCG', 'GCC', 'CAA', 'CAT', 'CAG', 'CAC', 'CTA', 'CTT', 'CTG', 'CTC',
                  'CGA', 'CGT', 'CGG', 'CGC', 'CCA', 'CCT', 'CCG', 'CCC'}

    totRefCods = 0

    # Change this when handling other organisms!!! This is for human
    # Codon frequncy by 1000 codons in the genome
    codonFreqsLit = {'TTT': 17.14, 'TCT': 16.93, 'TAT': 12.11, 'TGT': 10.40, 'TTC': 17.48, 'TCC': 17.32, 'TAC': 13.49,
                     'TGC': 10.81, 'TTA': 8.71, 'TCA': 14.14, 'TAA': 0.44, 'TGA': 0.79, 'TTG': 13.44, 'TCG': 4.03,
                     'TAG': 0.35, 'TGG': 11.60, 'CTT': 14.08, 'CCT': 19.31, 'CAT': 11.83, 'CGT': 4.55, 'CTC': 17.81,
                     'CCC': 19.11, 'CAC': 14.65, 'CGC': 8.71, 'CTA': 7.44, 'CCA': 18.92, 'CAA': 14.06, 'CGA': 6.42,
                     'CTG': 36.10, 'CCG': 6.22, 'CAG': 35.53, 'CGG': 10.79, 'ATT': 16.48, 'ACT': 14.26, 'AAT': 18.43,
                     'AGT': 14.05, 'ATC': 18.67, 'ACC': 17.85, 'AAC': 18.30, 'AGC': 19.69, 'ATA': 8.08, 'ACA': 16.52,
                     'AAA': 27.48, 'AGA': 13.28, 'ATG': 21.53, 'ACG': 5.59, 'AAG': 31.77, 'AGG': 12.13, 'GTT': 11.74,
                     'GCT': 18.99, 'GAT': 24.03, 'GGT': 10.83, 'GTC': 13.44, 'GCC': 25.84, 'GAC': 24.27, 'GGC': 19.79,
                     'GTA': 7.66, 'GCA': 17.04, 'GAA': 33.65, 'GGA': 17.12, 'GTG': 25.87, 'GCG': 5.91, 'GAG': 39.67,
                     'GGG': 15.35}

    codonCounter = {'TTT': 0, 'TCT': 0, 'TAT': 0, 'TGT': 0, 'TTC': 0, 'TCC': 0, 'TAC': 0,
                    'TGC': 0, 'TTA': 0, 'TCA': 0, 'TAA': 0, 'TGA': 0, 'TTG': 0, 'TCG': 0,
                    'TAG': 0, 'TGG': 0, 'CTT': 0, 'CCT': 0, 'CAT': 0, 'CGT': 0, 'CTC': 0,
                    'CCC': 0, 'CAC': 0, 'CGC': 0, 'CTA': 0, 'CCA': 0, 'CAA': 0, 'CGA': 0,
                    'CTG': 0, 'CCG': 0, 'CAG': 0, 'CGG': 0, 'ATT': 0, 'ACT': 0, 'AAT': 0,
                    'AGT': 0, 'ATC': 0, 'ACC': 0, 'AAC': 0, 'AGC': 0, 'ATA': 0, 'ACA': 0,
                    'AAA': 0, 'AGA': 0, 'ATG': 0, 'ACG': 0, 'AAG': 0, 'AGG': 0, 'GTT': 0,
                    'GCT': 0, 'GAT': 0, 'GGT': 0, 'GTC': 0, 'GCC': 0, 'GAC': 0, 'GGC': 0,
                    'GTA': 0, 'GCA': 0, 'GAA': 0, 'GGA': 0, 'GTG': 0, 'GCG': 0, 'GAG': 0,
                    'GGG': 0}

    aaCodonVecs = {
        'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
        'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
        'C': ['TGT', 'TGC'],
        'W': ['TGG'],
        'E': ['GAA', 'GAG'],
        'D': ['GAT', 'GAC'],
        'P': ['CCT', 'CCC', 'CCA', 'CCG'],
        'V': ['GTT', 'GTC', 'GTA', 'GTG'],
        'N': ['AAT', 'AAC'],
        'M': ['ATG'],
        'K': ['AAA', 'AAG'],
        'Y': ['TAT', 'TAC'],
        'I': ['ATT', 'ATC', 'ATA'],
        'Q': ['CAA', 'CAG'],
        'F': ['TTT', 'TTC'],
        'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
        'T': ['ACT', 'ACC', 'ACA', 'ACG'],
        '*': ['TAA', 'TAG', 'TGA'],
        'A': ['GCT', 'GCC', 'GCA', 'GCG'],
        'G': ['GGT', 'GGC', 'GGA', 'GGG'],
        'H': ['CAT', 'CAC']}

    # TODO: Add feature to weigh results with transcript abundance
    # define rareCodons
    codingRegions = []
    for gene in refGenes:
        for iso in gene.isoforms:
            codingSeqInQ = iso.codingSeq
            codingRegions.append(codingSeqInQ)
            maxC = len(codingSeqInQ) / 3
            i = 0
            while i < maxC:
                a = i * 3
                codonInQ = codingSeqInQ[a:a + 2]
                totRefCods += 1
                codonCounter[codonInQ] += 1
                i += 1

    # define aaFreqs
    AAs = aaCodonVecs.keys()
    AAFreqs = []
    rareCodons = []
    for aa in AAs:
        countingCods = aaCodonVecs[aa]
        aaFreq = 0
        for cod in countingCods:
            aaFreq += codonCounter[cod] / totRefCods
        AAFreqs.append([aa, aaFreq])

    # define rareCodons based on relative frequencies
    for i in range(len(AAFreqs)):
        aaFreqInQ = AAFreqs[i][1]
        aaInQ = AAFreqs[i][0]
        for cod in aaCodonVecs[aaInQ]:
            codFreq = codonCounter[cod] / totRefCods
            if codFreq / aaFreqInQ < 0.1:
                rareCodons.append(cod)

    for cod in codonTable:
        codonCounter[cod] = codonCounter[cod] / totRefCods
    print("______________________________________________________")
    print("Rare Codon Analysis:")
    print("_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_")
    print("The rare codons for ", org, " are: ", rareCodons)
    print("The total number of codons is: ", totRefCods)
    print("The codon frequencies are: ", codonCounter)
    print("The amino acid frequencies are: ", AAFreqs)
    print("______________________________________________________")

    return rareCodons, AAFreqs, codonCounter, totRefCods

rareCodons, AAFreqs, codonCounter, totRefCods = defineRareCodonsRefGenes('human')


#Total Codon Usage

In [ ]:
#Run total codon usage
def defineCodonUsageRefGB(org, trans: str):
    root = os.path.join('generider_refgenes', org)
    genes = loadTranscriptomeGeneBody(root)
    totalCodons = 0
    blank = {'+50bpUpstream': 0, '-50bpDownstream': 0, 'Exon': 0, '-50bpExon': 0, '+50bpExon': 0, 'Splice': 0}
    codonCounter = {'TTT': 0, 'TCT': 0, 'TAT': 0, 'TGT': 0, 'TTC': 0, 'TCC': 0, 'TAC': 0,
                    'TGC': 0, 'TTA': 0, 'TCA': 0, 'TAA': 0, 'TGA': 0, 'TTG': 0, 'TCG': 0,
                    'TAG': 0, 'TGG': 0, 'CTT': 0, 'CCT': 0, 'CAT': 0, 'CGT': 0, 'CTC': 0,
                    'CCC': 0, 'CAC': 0, 'CGC': 0, 'CTA': 0, 'CCA': 0, 'CAA': 0, 'CGA': 0,
                    'CTG': 0, 'CCG': 0, 'CAG': 0, 'CGG': 0, 'ATT': 0, 'ACT': 0, 'AAT': 0,
                    'AGT': 0, 'ATC': 0, 'ACC': 0, 'AAC': 0, 'AGC': 0, 'ATA': 0, 'ACA': 0,
                    'AAA': 0, 'AGA': 0, 'ATG': 0, 'ACG': 0, 'AAG': 0, 'AGG': 0, 'GTT': 0,
                    'GCT': 0, 'GAT': 0, 'GGT': 0, 'GTC': 0, 'GCC': 0, 'GAC': 0, 'GGC': 0,
                    'GTA': 0, 'GCA': 0, 'GAA': 0, 'GGA': 0, 'GTG': 0, 'GCG': 0, 'GAG': 0,
                    'GGG': 0}

    codonFreqsLit = {'TTT': 17.14, 'TCT': 16.93, 'TAT': 12.11, 'TGT': 10.40, 'TTC': 17.48, 'TCC': 17.32, 'TAC': 13.49,
                     'TGC': 10.81, 'TTA': 8.71, 'TCA': 14.14, 'TAA': 0.44, 'TGA': 0.79, 'TTG': 13.44, 'TCG': 4.03,
                     'TAG': 0.35, 'TGG': 11.60, 'CTT': 14.08, 'CCT': 19.31, 'CAT': 11.83, 'CGT': 4.55, 'CTC': 17.81,
                     'CCC': 19.11, 'CAC': 14.65, 'CGC': 8.71, 'CTA': 7.44, 'CCA': 18.92, 'CAA': 14.06, 'CGA': 6.42,
                     'CTG': 36.10, 'CCG': 6.22, 'CAG': 35.53, 'CGG': 10.79, 'ATT': 16.48, 'ACT': 14.26, 'AAT': 18.43,
                     'AGT': 14.05, 'ATC': 18.67, 'ACC': 17.85, 'AAC': 18.30, 'AGC': 19.69, 'ATA': 8.08, 'ACA': 16.52,
                     'AAA': 27.48, 'AGA': 13.28, 'ATG': 21.53, 'ACG': 5.59, 'AAG': 31.77, 'AGG': 12.13, 'GTT': 11.74,
                     'GCT': 18.99, 'GAT': 24.03, 'GGT': 10.83, 'GTC': 13.44, 'GCC': 25.84, 'GAC': 24.27, 'GGC': 19.79,
                     'GTA': 7.66, 'GCA': 17.04, 'GAA': 33.65, 'GGA': 17.12, 'GTG': 25.87, 'GCG': 5.91, 'GAG': 39.67,
                     'GGG': 15.35}

    totalProfile = {'+50bpUpstream': 0, '-50bpDownstream': 0, 'Exon': 0, '-50bpExon': 0, '+50bpExon': 0, 'Splice': 0}

    codLoc = {'TTT': blank, 'TCT': blank, 'TAT': blank, 'TGT': blank, 'TTC': blank, 'TCC': blank, 'TAC': blank,
              'TGC': blank, 'TTA': blank, 'TCA': blank, 'TAA': blank, 'TGA': blank, 'TTG': blank, 'TCG': blank,
              'TAG': blank, 'TGG': blank, 'CTT': blank, 'CCT': blank, 'CAT': blank, 'CGT': blank, 'CTC': blank,
              'CCC': blank, 'CAC': blank, 'CGC': blank, 'CTA': blank, 'CCA': blank, 'CAA': blank, 'CGA': blank,
              'CTG': blank, 'CCG': blank, 'CAG': blank, 'CGG': blank, 'ATT': blank, 'ACT': blank, 'AAT': blank,
              'AGT': blank, 'ATC': blank, 'ACC': blank, 'AAC': blank, 'AGC': blank, 'ATA': blank, 'ACA': blank,
              'AAA': blank, 'AGA': blank, 'ATG': blank, 'ACG': blank, 'AAG': blank, 'AGG': blank, 'GTT': blank,
              'GCT': blank, 'GAT': blank, 'GGT': blank, 'GTC': blank, 'GCC': blank, 'GAC': blank, 'GGC': blank,
              'GTA': blank, 'GCA': blank, 'GAA': blank, 'GGA': blank, 'GTG': blank, 'GCG': blank, 'GAG': blank,
              'GGG': blank}

    aaFreqs = {'S': 0, 'L': 0, 'C': 0, 'W': 0, 'E': 0, 'D': 0, 'P': 0, 'V': 0, 'N': 0, 'M': 0, 'K': 0, 'Y': 0,
               'I': 0, 'Q': 0, 'F': 0, 'R': 0, 'T': 0, '*': 0, 'A': 0, 'G': 0, 'H': 0}

    codonProfilePerGene = []
    for gene in genes:
        for isoGB in gene.isoforms:
            for aa in isoGB.associatedProtein.aaSeq:
                aaFreqs[aa] += 1
            codons = codonVecFromGenSeqGB(isoGB)
            codonLocation = codLoc
            for codon, loc in codons:
                codonCounter[codon] += 1
                totalCodons += 1
                totalProfile[loc] += 1
                codonLocation[codon][loc] += 1

            codonProfilePerGene.append(codonLocation)

    for aa in aaFreqs.keys():
        aaFreqs[aa] = aaFreqs[aa] / totalCodons

    codonUsageVec = {'TTT': blank, 'TCT': blank, 'TAT': blank, 'TGT': blank, 'TTC': blank, 'TCC': blank, 'TAC': blank,
              'TGC': blank, 'TTA': blank, 'TCA': blank, 'TAA': blank, 'TGA': blank, 'TTG': blank, 'TCG': blank,
              'TAG': blank, 'TGG': blank, 'CTT': blank, 'CCT': blank, 'CAT': blank, 'CGT': blank, 'CTC': blank,
              'CCC': blank, 'CAC': blank, 'CGC': blank, 'CTA': blank, 'CCA': blank, 'CAA': blank, 'CGA': blank,
              'CTG': blank, 'CCG': blank, 'CAG': blank, 'CGG': blank, 'ATT': blank, 'ACT': blank, 'AAT': blank,
              'AGT': blank, 'ATC': blank, 'ACC': blank, 'AAC': blank, 'AGC': blank, 'ATA': blank, 'ACA': blank,
              'AAA': blank, 'AGA': blank, 'ATG': blank, 'ACG': blank, 'AAG': blank, 'AGG': blank, 'GTT': blank,
              'GCT': blank, 'GAT': blank, 'GGT': blank, 'GTC': blank, 'GCC': blank, 'GAC': blank, 'GGC': blank,
              'GTA': blank, 'GCA': blank, 'GAA': blank, 'GGA': blank, 'GTG': blank, 'GCG': blank, 'GAG': blank,
              'GGG': blank}

    #get an average of where the codons are found accounting for all the genes analyzed
    median = {'+50bpUpstream': 0,
              '-50bpDownstream': 0,
              'Exon': 0,
              '-50bpExon': 0,
              '+50bpExon': 0,
              'Splice': 0}

    for code in codLoc.keys():
        m = median
        for kee in median.keys():
            for cod, diction in codonProfilePerGene:
                if cod == code:
                    for d in diction.keys():
                        if kee in d:
                            #Add a fraction representing the codon's presence in the location per 1000 codons
                            m[kee] += diction[d] * 1000 / len([c for c, di in codonProfilePerGene if di == d])
        codonUsageVec[code] = m

    codonUseScoreByGeneLit = []
    for gene in genes:
        for iso in gene.isoforms:
            useScore = 0
            for i in range(len(iso.codingSeq)/3):
                codon = iso.codingSeq[i*3:i*3+2]
                useScore += codonFreqsLit[codon]
            codonUseScoreByGeneLit.append(useScore / (len(iso.codingSeq)/3))


    #Roll analysis into a class instance
    codAna = CodonAnalysis(org, aaFreqs, totalCodons, codonUsageVec, codonFreqsLit, codonProfilePerGene, codonUseScoreByGeneLit)

    #If the transcriptome directory does not exist, make it
    root = os.path.join('generider_heuristics', org)
    if not os.path.exists('generider_heuristics'):
        os.makedirs('generider_heuristics')
    #Save the analysis
    print("Saving codon analysis for ", org, " to ", root)
    if not os.path.exists(root):
        os.makedirs(root)
    print(codAna)
    raise NotImplementedError
    #Save
    with open(root, 'wb') as roo:
        pickle.dump(codAna, roo)

#Codon Pair Bias

In [ ]:
#Run codon pair bias


In [ ]:
#Run G/C dynamics


In [ ]:
#Run k-mer


In [ ]:
#Run physics


In [ ]:
#Save run

#Compile tests

with open(refgenespth, 'w') as f:
  f.write('test')